# Figures for JOSS Paper

### Load packages

Import Python packages that are used for the analysis

In [ ]:
import datacube
import xarray as xr
import contextily as ctx
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from odc.geo.xr import assign_crs
from matplotlib.colors import ListedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import sys
sys.path.insert(1, "../Tools/")
from dea_tools.plotting import rgb
from dea_tools.validation import xr_random_sampling
from dea_tools.spatial import xr_vectorize, xr_rasterize
from dea_tools.landcover import lc_colourmap, get_colour_scheme, make_colourbar

## Connect to the datacube

Connect to the datacube so we can access DEA data.

In [ ]:
dc = datacube.Datacube(app="Random_sampling")

## Analysis parameters

* `central_lat`: Central latitude for the study area (e.g. `-35.29`). Set this to the approximate centre of your area of interest.
* `central_lon`: Central longitude for the study area (e.g. `149.113`). Set this to the approximate centre of your area of interest.
* `buffer`: Distance in degrees to load around the central point (e.g. `0.3`). Controls the size of the spatial bounding box—larger buffers will load a bigger area and may increase processing time.
* `time`: Time range for analysis as a tuple of start and end dates in `YYYY-MM-DD` format (e.g. `("2023-01-01", "2023-12-31")`).


In [ ]:
# Set the central latitude and longitude
central_lat = -34.819 
central_lon =  138.699

# Set the buffer to load around the central coordinates. 
buffer = 0.3

#time range to load
time = ("2023-01-01", "2023-12-31")

# Compute the bounding box for the study area
latitude = (central_lat - buffer, central_lat + buffer)
longitude = (central_lon - buffer, central_lon + buffer)

## Load DEA Land Cover data

We will also load a geomedian from the same period for visualisation purposes.

In [ ]:
# Load DEA Land Cover data
lc = dc.load(
    product="ga_ls_landcover_class_cyear_3",
    output_crs="EPSG:3577",
    measurements=["level3"],
    x=longitude,
    y=latitude,
    resolution=(-30, 30),
    time=time,
)

# Convert to datarray and mask any no-data
lc = lc["level3"].squeeze()
lc = lc.where(lc != 255)

# Also load a geomedian
ds = dc.load(
    product="ga_ls8cls9c_gm_cyear_3",
    time=time,
    measurements=["nbart_red", "nbart_green", "nbart_blue"],
    x=longitude,
    y=latitude,
    output_crs="EPSG:3577",
    resolution=(30, -30),
).squeeze()

## Extract random samples from DEA Land Cover

Using the different sampling strategies outlined in the Description section above.

In [ ]:
# Equal stratified random samples
train_points_equal_stratified_random = xr_random_sampling(
    lc, sampling="equal_stratified_random", n=400, verbose=False
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

# Trim no-data off the colormap
colour_scheme = get_colour_scheme("level3")
cmap, norm = lc_colourmap(colour_scheme)
im = lc.plot(cmap=cmap, norm=norm, ax=ax[0], add_labels=False, add_colorbar=False)
make_colourbar(fig, ax[0], measurement="level3", labelsize=7, horizontal=False)
ax[0].set_title("DEA Land Cover", fontsize=12)

cmap=ListedColormap(cmap.colors[:-1])

ds[["nbart_red", "nbart_green", "nbart_blue"]].to_array().plot.imshow(
    robust=True, ax=ax[1], add_labels=False)
train_points_equal_stratified_random.plot(ax=ax[1], column="class", cmap=cmap, legend=False, categorical=True)
ax[1].set_title("Equal stratified random sampling", fontsize=12);

for a in ax.ravel():
    a.axes.get_xaxis().set_ticks([])
    a.axes.get_yaxis().set_ticks([]);

plt.savefig("figures/xr_random_sampling.png", dpi=300, bbox_inches="tight")

## RGB plots

In [ ]:
# Set the central latitude and longitude
central_lat = -17.549172 
central_lon = 140.865

# Set the buffer to load around the central coordinates. 
buffer = 0.075

# Compute the bounding box for the study area
latitude = (central_lat - buffer, central_lat + buffer)
longitude = (central_lon - buffer, central_lon + buffer)

dc = datacube.Datacube(app="abc")

# Set up a datacube query to load data for
query = {
    "x": longitude,
    "y": latitude,
    "time": ("2023", "2024"),
    "measurements": ["nbart_red", "nbart_green", "nbart_blue", "nbart_nir", "nbart_swir_2"],
}

# Load available satellite data from Landsat 8 geomedian product
ds = dc.load(product="ga_ls8cls9c_gm_cyear_3", **query)

In [ ]:
fig,ax = plt.subplots(1,3, figsize=(15,5), sharey=True, layout='constrained')

ds[["nbart_red", "nbart_green", "nbart_blue"]].isel(time=-1).to_array().plot.imshow(ax=ax[0], robust=True, add_labels=False)
ds[["nbart_swir_2", "nbart_nir", "nbart_green"]].isel(time=-1).to_array().plot.imshow(ax=ax[1], robust=True, add_labels=False)
ds[["nbart_nir", "nbart_red", "nbart_green"]].isel(time=-1).to_array().plot.imshow(ax=ax[2], robust=True, add_labels=False)

for a in ax.ravel():
    a.axes.get_xaxis().set_ticks([])
    a.axes.get_yaxis().set_ticks([]);
    a.set_title(None)

plt.savefig("figures/RGB_images.png", dpi=300, bbox_inches="tight")

## Rasterize and vectorise

In [ ]:
dc = datacube.Datacube(app='Rasterize_vectorize')

In [ ]:
# Create a query object
query = {
    'x': (142.1, 142.80),
    'y': (-32.1, -32.6),
    # 'time': ('2024')
}

# Load WoFS through the datacube
ds = dc.load(product='wofs_summary', 
             **query)

ds = ds.where(ds>=0)

ds = ds.where(ds<=1)

In [ ]:
# Define 10 equal bins between 0 and 1 for reclassification
bins = np.arange(0, 1.01, 0.1) # 11 edges for 10 bins

# Digitize values into bins (1–10)
binned = xr.apply_ufunc(
    np.digitize,
    ds.frequency,
    kwargs={"bins": bins, "right": False},
    dask="allowed",
)

# Optional: convert bin numbers to 0–9 range
binned = binned - 0.10

binned.name = "Frequency"
binned = binned / 10
binned=binned.squeeze()


In [ ]:
gdf = xr_vectorize(da=binned,
                   attribute_col='Frequency'
                  )

# enforce bins because geopandas automatically derives the color bins
# from the unique values present in the data. Also need to remove the lowest
# 0-0.1 value so areas without water aren't included in the plot
gdf['Frequency'] = gdf['Frequency'].mask(gdf['Frequency'] <= 0.1, np.nan)

# Example categorical breaks
labels = [round(b, 1) for b in bins[:-1]]  # category labels

# Assign Frequency to fixed bins (even if some bins have no data)
gdf["Frequency_bin"] = pd.cut(
    gdf["Frequency"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Make the categorical dtype explicit
gdf["Frequency_bin"] = gdf["Frequency_bin"].astype(
    pd.CategoricalDtype(categories=labels, ordered=True)
)

In [ ]:
fig,ax = plt.subplots(1,2, figsize=(11,5), sharey=True, layout='constrained')
cmap='gnuplot'

im = binned.where(binned>=0.1).plot(ax=ax[0], add_labels=False, cmap=cmap, add_colorbar=False, vmin=0, vmax=1)
ax[0].set_title('Water frequency as pixel-based DataArray')

cmap = plt.get_cmap(cmap, len(labels))
cmap = ListedColormap([cmap(i) for i in range(len(labels))])

# Plot
gdf.plot(
    ax=ax[1],
    column="Frequency_bin",
    cmap=cmap,
    categorical=True,
    legend=True,
    edgecolor="black",
    linewidth=0.25,
    legend_kwds={'ncols': 1, 'loc': 'lower right'}
)

ax[1].set_title('Water frequency as vectorised GeoDataFrame')

xmin, ymin, xmax, ymax = gdf.total_bounds
ax[1].set_xlim(xmin, xmax)
ax[1].set_ylim(ymin, ymax)

for a in ax.ravel():
    a.axes.get_xaxis().set_ticks([])
    a.axes.get_yaxis().set_ticks([]);

axins1 = inset_axes(ax[0], width="65%",height="5%",loc="lower right", borderpad=2)
cbar = fig.colorbar(im, cax=axins1, orientation='horizontal', spacing='proportional')
cbar.ax.set_title('Water Frequency', fontsize=10);

plt.savefig("figures/rasterize_vectorize.png", dpi=300, bbox_inches="tight")